### Combined Ollama Installation and Startup

This cell combines the previous Ollama installation and server startup logic for easier debugging. It includes:

1.  **Downloading and Installing Ollama:** Fetches the latest GPU-enabled Ollama release and extracts it to `/usr/local/ollama_bin`.
2.  **Robust Path Discovery:** Carefully locates the `ollama` executable and its `llama-server` component within the installation directory.
3.  **Environment Variable Setup:** Configures `PATH`, `OLLAMA_HOST`, `LD_LIBRARY_PATH`, and explicitly unsets `OLLAMA_LLM_LIBRARY` to prevent conflicts.
4.  **Ollama Server Startup:** Starts the Ollama server, changing the working directory to help it resolve internal paths.
5.  **Model Pulling and Aliasing:** Pulls the `qwen2.5-coder:7b` model and creates an alias.
6.  **GPU Status Verification:** Checks the Ollama server logs to determine if GPU is being utilized.

This Notebook for Collab and GPU if you can access GPU quota.

In [ ]:
import subprocess, time, os, json
import glob # For more robust path finding

print("--- Starting Combined Ollama Setup ---")

# --- Ollama Installation (from original cell a68669237-c5b5-47fe-8aea-41b0b94e38df) ---
print("Fetching latest Ollama GPU release URL...")

# Fetch the latest release information from GitHub API
try:
    release_info = subprocess.run(
        "curl -s https://api.github.com/repos/ollama/ollama/releases/latest",
        shell=True, capture_output=True, text=True, check=True
    ).stdout
    latest_release = json.loads(release_info)
except (subprocess.CalledProcessError, json.JSONDecodeError) as e:
    raise RuntimeError(f"Failed to fetch latest Ollama release info: {e}")

# Try to find the standard linux-amd64 .tar.zst archive
download_url = None
ollama_archive_filename = "ollama-linux-amd64.tar.zst" # Standard AMD64 Linux build
for asset in latest_release.get('assets', []):
    asset_name = asset.get('name', '')
    asset_url = asset.get('browser_download_url', '')
    if ollama_archive_filename == asset_name:
        download_url = asset_url
        print(f"Found standard Linux AMD64 archive: {ollama_archive_filename}")
        break

if not download_url:
    raise RuntimeError(f"Could not find the standard Linux AMD64 Ollama download URL with name '{ollama_archive_filename}' in the latest release. Please check the GitHub releases manually.")

print(f"Downloading Ollama GPU version from: {download_url}")

# Download Ollama archive with retries
MAX_RETRIES = 3
for attempt in range(MAX_RETRIES):
    try:
        subprocess.run(
            f"curl -fsSL {download_url} -o /tmp/ollama.tar.zst",
            shell=True, check=True
        )
        print("Download successful!")
        break # Exit loop if download is successful
    except subprocess.CalledProcessError as e:
        print(f"Download attempt {attempt + 1}/{MAX_RETRIES} failed: {e}")
        if attempt < MAX_RETRIES - 1:
            time.sleep(5) # Wait before retrying
        else:
            raise # Re-raise the exception after all retries fail

# Install zstd
subprocess.run("apt-get install -y zstd -q", shell=True, capture_output=True, check=True)

# Define the Ollama installation root directory
OLLAMA_INSTALL_ROOT = "/usr/local/ollama_bin" # A clean, dedicated directory for the Ollama binaries
os.makedirs(OLLAMA_INSTALL_ROOT, exist_ok=True)
print(f"Created Ollama installation root directory: {OLLAMA_INSTALL_ROOT}")

# Extract Ollama to the dedicated directory
subprocess.run(f"tar --use-compress-program=unzstd -xf /tmp/ollama.tar.zst -C {OLLAMA_INSTALL_ROOT}/", shell=True, check=True)

# Attempt to find the ollama executable more robustly
OLLAMA_EXECUTABLE_PATH = None
OLLAMA_BIN_DIR = None

# Case 1: 'ollama' executable is directly in OLLAMA_INSTALL_ROOT
if os.path.exists(os.path.join(OLLAMA_INSTALL_ROOT, 'ollama')):
    OLLAMA_EXECUTABLE_PATH = os.path.join(OLLAMA_INSTALL_ROOT, 'ollama')
    OLLAMA_BIN_DIR = OLLAMA_INSTALL_ROOT
    print(f"Ollama executable found directly in root: {OLLAMA_EXECUTABLE_PATH}")
else:
    # Case 2: 'ollama' executable might be in a subdirectory (e.g., OLLAMA_INSTALL_ROOT/ollama/ollama)
    print(f"Ollama executable not found directly in {OLLAMA_INSTALL_ROOT}. Searching subdirectories...")
    ollama_exec_candidates = glob.glob(os.path.join(OLLAMA_INSTALL_ROOT, '**', 'ollama'), recursive=True)
    executable_candidates = [p for p in ollama_exec_candidates if os.path.isfile(p) and os.access(p, os.X_OK)]

    if executable_candidates:
        # Prioritize candidates directly under OLLAMA_INSTALL_ROOT or shallowest path
        executable_candidates.sort(key=lambda x: x.count(os.sep)) # Sort by path depth
        OLLAMA_EXECUTABLE_PATH = executable_candidates[0]
        OLLAMA_BIN_DIR = os.path.dirname(OLLAMA_EXECUTABLE_PATH)
        print(f"Ollama executable found at: {OLLAMA_EXECUTABLE_PATH}")
    else:
        raise RuntimeError(f"Ollama executable not found in {OLLAMA_INSTALL_ROOT} or its subdirectories after extraction.")

# Ensure we have valid paths
if not OLLAMA_EXECUTABLE_PATH or not OLLAMA_BIN_DIR:
    raise RuntimeError("Failed to determine Ollama executable path and binary directory.")

print(f"Ollama binaries directory: {OLLAMA_BIN_DIR}")

# Find the llama-server binary relative to OLLAMA_INSTALL_ROOT
# It is usually in the 'lib' directory directly under the main installation root.
LLAMA_SERVER_PATH = os.path.join(OLLAMA_INSTALL_ROOT, 'lib', 'llama-server')
if not os.path.exists(LLAMA_SERVER_PATH):
    # If not found at the most common location, broaden search within OLLAMA_INSTALL_ROOT
    print(f"llama-server not found at expected path {LLAMA_SERVER_PATH}. Searching broader within {OLLAMA_INSTALL_ROOT}...")
    llama_server_candidates = glob.glob(os.path.join(OLLAMA_INSTALL_ROOT, '**', 'llama-server'), recursive=True)
    executable_llama_server_candidates = [p for p in llama_server_candidates if os.path.isfile(p) and os.access(p, os.X_OK)]

    if executable_llama_server_candidates:
        LLAMA_SERVER_PATH = executable_llama_server_candidates[0]
        print(f"llama-server binary found at: {LLAMA_SERVER_PATH} (after broad search)")
    else:
        raise RuntimeError(f"llama-server binary not found within {OLLAMA_INSTALL_ROOT} or its subdirectories.")

subprocess.run(f"chmod +x {LLAMA_SERVER_PATH}", shell=True, check=True)
print(f"llama-server binary found and made executable at: {LLAMA_SERVER_PATH}")

# Ensure the ollama executable is executable
subprocess.run(f"chmod +x {OLLAMA_EXECUTABLE_PATH}", shell=True, check=True)

# Add the directory containing ollama and its companion binaries to PATH
os.environ["PATH"] = f"{OLLAMA_BIN_DIR}:{os.environ.get('PATH', '')}"
print(f"Added '{OLLAMA_BIN_DIR}' to PATH environment variable.")

# Verify the Ollama version using the custom executable path explicitly
v = subprocess.run([OLLAMA_EXECUTABLE_PATH, "--version"], capture_output=True, text=True, check=True)
print("✅ Ollama:", v.stdout.strip())

# Store the path to the executable and its bin directory for later use
os.environ["OLLAMA_CUSTOM_EXECUTABLE_PATH"] = OLLAMA_EXECUTABLE_PATH
os.environ["OLLAMA_CUSTOM_BIN_DIR"] = OLLAMA_BIN_DIR
os.environ["OLLAMA_LLAMA_SERVER_PATH"] = LLAMA_SERVER_PATH

# --- Ollama Server Startup (from original cell 351d8fe6) ---

# Kill any existing Ollama processes to free up the port
print("Killing any existing Ollama processes...")
subprocess.run("pkill ollama 2>/dev/null", shell=True)
time.sleep(2) # Give a moment for the process to terminate

# Create a copy of the current environment variables
env_vars = os.environ.copy()

# Explicitly set the GPU-related environment variables
env_vars["OLLAMA_HOST"] = "0.0.0.0:11434"
env_vars["LD_LIBRARY_PATH"] = "/usr/local/cuda/lib64:" + env_vars.get("LD_LIBRARY_PATH", "")

# Explicitly UNSET OLLAMA_LLM_LIBRARY if it's present, to prevent interference
if "OLLAMA_LLM_LIBRARY" in env_vars:
    print(f"Explicitly unsetting OLLAMA_LLM_LIBRARY (was: {env_vars['OLLAMA_LLM_LIBRARY']}) to prevent conflicts.")
    del env_vars["OLLAMA_LLM_LIBRARY"]

env_vars["OLLAMA_DEBUG"] = "1" # Enable debug logging for Ollama

print("Starting Ollama server with environment variables:")
for k, v in env_vars.items():
    if "OLLAMA" in k or "LD_LIBRARY_PATH" in k or "PATH" in k:
        print(f"  {k}: {v}")

# Use the executable directly, which is now correctly set in OLLAMA_CUSTOM_EXECUTABLE_PATH
ollama_exec = env_vars.get("OLLAMA_CUSTOM_EXECUTABLE_PATH")
if not ollama_exec or not os.path.exists(ollama_exec):
    raise RuntimeError(f"Ollama executable path not correctly set or found: {ollama_exec}")

ollama_bin_dir = env_vars.get("OLLAMA_CUSTOM_BIN_DIR")
if not ollama_bin_dir or not os.path.exists(ollama_bin_dir):
    raise RuntimeError(f"Ollama binary directory not correctly set or found: {ollama_bin_dir}")


print(f"Changing current working directory to {ollama_bin_dir} to start Ollama server.")
current_dir = os.getcwd()
os.chdir(ollama_bin_dir)
try:
    subprocess.Popen(
        [ollama_exec, "serve"],
        stdout=open("/tmp/ollama_serve.log", "w"),
        stderr=subprocess.STDOUT,
        env=env_vars # Pass the explicitly constructed environment variables
    )
finally:
    os.chdir(current_dir) # Change back to original directory

time.sleep(30) # Increased sleep time to allow Ollama to fully start and load libraries

# Pull only if not already cached
ollama_exec_for_commands = env_vars.get("OLLAMA_CUSTOM_EXECUTABLE_PATH")
if not ollama_exec_for_commands or not os.path.exists(ollama_exec_for_commands):
    raise RuntimeError(f"Ollama executable path for commands not correctly set or found: {ollama_exec_for_commands}")

models_command = [ollama_exec_for_commands, "list"]
print(f"Running command: {' '.join(models_command)}")
models = subprocess.run(models_command, capture_output=True, text=True, env=env_vars) # Pass env_vars here too
print(models.stdout)
if models.returncode != 0:
    print(f"Error listing models: {models.stderr}")
    raise RuntimeError("Failed to list Ollama models.")


if "qwen2.5-coder" not in models.stdout:
    print("Pulling qwen2.5-coder:7b (~4GB)... This might take a while.")
    pull_command = [ollama_exec_for_commands, "pull", "qwen2.5-coder:7b"]
    subprocess.run(pull_command, check=True, env=env_vars)
else:
    print("✅ Model already cached, skipping download")

# Create claude- alias
cp_command = [ollama_exec_for_commands, "cp", "qwen2.5-coder:7b", "claude-qwen"]
subprocess.run(cp_command, capture_output=True, check=True, env=env_vars)
print("✅ Ollama ready")

# Verify Ollama log for GPU usage
print("--- Checking Ollama server log for GPU status ---")
log_output = subprocess.run(['tail', '-n', '50', '/tmp/ollama_serve.log'], capture_output=True, text=True).stdout
print(log_output)
if 'id=gpu' in log_output:
    print("✅ Ollama is likely using the GPU!")
elif 'id=cpu' in log_output:
    print("❌ Ollama is still using the CPU. Further investigation needed.")
else:
    print("⚠️ Could not determine Ollama GPU status from logs.")
print("--- Combined Ollama Setup Complete ---")

--- Starting Combined Ollama Setup ---
Fetching latest Ollama GPU release URL...
Found standard Linux AMD64 archive: ollama-linux-amd64.tar.zst
[15:36:49] ✅ #2 | Models: ['claude-qwen:latest', 'qwen2.5-coder:7b']
Download successful!
Created Ollama installation root directory: /usr/local/ollama_bin
Ollama executable not found directly in /usr/local/ollama_bin. Searching subdirectories...
Ollama executable found at: /usr/local/ollama_bin/bin/ollama
Ollama binaries directory: /usr/local/ollama_bin/bin
llama-server not found at expected path /usr/local/ollama_bin/lib/llama-server. Searching broader within /usr/local/ollama_bin...
llama-server binary found at: /usr/local/ollama_bin/lib/ollama/llama-server (after broad search)
llama-server binary found and made executable at: /usr/local/ollama_bin/lib/ollama/llama-server
Added '/usr/local/ollama_bin/bin' to PATH environment variable.
✅ Ollama: ollama version is 0.30.6
Killing any existing Ollama processes...
Starting Ollama server with envi

In [ ]:
# import subprocess, time, os

# # Kill any existing Ollama processes to free up the port
# print("Killing any existing Ollama processes...")
# subprocess.run("pkill ollama 2>/dev/null", shell=True)
# time.sleep(2) # Give a moment for the process to terminate

# # Create a copy of the current environment variables
# env_vars = os.environ.copy()

# # Explicitly set the GPU-related environment variables
# env_vars["OLLAMA_HOST"] = "0.0.0.0:11434"
# env_vars["LD_LIBRARY_PATH"] = "/usr/local/cuda/lib64:" + env_vars.get("LD_LIBRARY_PATH", "")

# # Removed: OLLAMA_LLM_LIBRARY setting, as it can interfere with Ollama's internal path resolution.
# # Ollama is expected to find 'llama-server' relative to its executable, especially after changing CWD.
# print("OLLAMA_LLM_LIBRARY setting removed to rely on Ollama's default discovery.")

# env_vars["OLLAMA_DEBUG"] = "1" # Enable debug logging for Ollama

# print("Starting Ollama server with environment variables:")
# for k, v in env_vars.items():
#     if "OLLAMA" in k or "LD_LIBRARY_PATH" in k or "PATH" in k: # Print relevant variables for debugging
#         print(f"  {k}: {v}")

# # Use the executable directly, which is now correctly set in OLLAMA_CUSTOM_EXECUTABLE_PATH
# ollama_exec = env_vars.get("OLLAMA_CUSTOM_EXECUTABLE_PATH")
# if not ollama_exec or not os.path.exists(ollama_exec):
#     raise RuntimeError(f"Ollama executable path not correctly set or found: {ollama_exec}")

# ollama_bin_dir = env_vars.get("OLLAMA_CUSTOM_BIN_DIR")
# if not ollama_bin_dir or not os.path.exists(ollama_bin_dir):
#     raise RuntimeError(f"Ollama binary directory not correctly set or found: {ollama_bin_dir}")


# print(f"Changing current working directory to {ollama_bin_dir} to start Ollama server.")
# current_dir = os.getcwd()
# os.chdir(ollama_bin_dir)
# try:
#     subprocess.Popen(
#         [ollama_exec, "serve"],
#         stdout=open("/tmp/ollama_serve.log", "w"),
#         stderr=subprocess.STDOUT,
#         env=env_vars # Pass the explicitly constructed environment variables
#     )
# finally:
#     os.chdir(current_dir) # Change back to original directory

# time.sleep(30) # Increased sleep time to allow Ollama to fully start and load libraries

# # Pull only if not already cached
# ollama_exec_for_commands = env_vars.get("OLLAMA_CUSTOM_EXECUTABLE_PATH")
# if not ollama_exec_for_commands or not os.path.exists(ollama_exec_for_commands):
#     raise RuntimeError(f"Ollama executable path for commands not correctly set or found: {ollama_exec_for_commands}")

# models_command = [ollama_exec_for_commands, "list"]
# print(f"Running command: {' '.join(models_command)}")
# models = subprocess.run(models_command, capture_output=True, text=True, env=env_vars) # Pass env_vars here too
# print(models.stdout)
# if models.returncode != 0:
#     print(f"Error listing models: {models.stderr}")
#     raise RuntimeError("Failed to list Ollama models.")


# if "qwen2.5-coder" not in models.stdout:
#     print("Pulling qwen2.5-coder:7b (~4GB)... This might take a while.")
#     pull_command = [ollama_exec_for_commands, "pull", "qwen2.5-coder:7b"]
#     subprocess.run(pull_command, check=True, env=env_vars)
# else:
#     print("✅ Model already cached, skipping download")

# # Create claude- alias
# cp_command = [ollama_exec_for_commands, "cp", "qwen2.5-coder:7b", "claude-qwen"]
# subprocess.run(cp_command, capture_output=True, check=True, env=env_vars)
# print("✅ Ollama ready")

# # Verify Ollama log for GPU usage
# print("--- Checking Ollama server log for GPU status ---")
# log_output = subprocess.run(['tail', '-n', '50', '/tmp/ollama_serve.log'], capture_output=True, text=True).stdout
# print(log_output)
# if 'id=gpu' in log_output:
#     print("✅ Ollama is likely using the GPU!")
# elif 'id=cpu' in log_output:
#     print("❌ Ollama is still using the CPU. Further investigation needed.")
# else:
#     print("⚠️ Could not determine Ollama GPU status from logs.")

In [ ]:
import subprocess, time

# Re-install litellm if needed
subprocess.run("pip install 'litellm[proxy]==1.82.4' -q", shell=True)

# Updated config to ensure the model name matches exactly what Claude terminal requests
config = """
model_list:
  - model_name: claude-sonnet-4-6
    litellm_params:
      model: ollama_chat/qwen2.5-coder:7b
      api_base: http://localhost:11434

litellm_settings:
  drop_params: true
  set_verbose: false
"""
with open("/tmp/litellm_config.yaml", "w") as f:
    f.write(config)

# Kill existing proxy
subprocess.run("pkill litellm 2>/dev/null", shell=True)
time.sleep(2)

print("Starting LiteLLM in background on port 8081...")
subprocess.Popen(
    "litellm --config /tmp/litellm_config.yaml --port 8081 > /tmp/litellm.log 2>&1 &",
    shell=True
)

# Wait and verify health
time.sleep(5)
print("Verifying local LiteLLM health...")
!curl -s http://localhost:8081/health

Starting LiteLLM in background on port 8081...
Verifying local LiteLLM health...


# Cloudflared Tunnel Log Analysis (No longer active as we are using ngrok)

# Verify Cloudflared Process (No longer active as we are using ngrok)

In [ ]:
import subprocess

# Using tail to read the last 100 lines of the log file
log_output = subprocess.run(['tail', '-n', '100', '/tmp/ollama_serve.log'], capture_output=True, text=True)
print(log_output.stdout)
if log_output.stderr:
    print("Error reading log file:", log_output.stderr)

time=2026-06-06T15:37:20.638Z level=INFO source=routes.go:1919 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_DEBUG:DEBUG OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://0.0.0.0:11434 OLLAMA_IGPU_ENABLE: OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/root/.ollama/models OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https://0.0.0.0 http://0.0.0.0:* h

In [ ]:
!pip install pyngrok -q

In [ ]:
import subprocess

# Ensure pyngrok is installed for the subsequent ngrok setup cell
subprocess.run("pip install pyngrok -q", shell=True)
print("✅ pyngrok installed.")

✅ pyngrok installed.


### Ngrok Setup
1. Get your token from [dashboard.ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken).
2. Replace the placeholder below with your token.

In [ ]:
# Ensure pyngrok is installed in the current kernel environment
!pip install pyngrok -q

import subprocess, time, sys
from pyngrok import ngrok

# Kill old tunnels and processes
subprocess.run("pkill cloudflared 2>/dev/null", shell=True)
ngrok.kill()
time.sleep(2)

# REPLACE WITH YOUR ACTUAL TOKEN
NGROK_TOKEN = "3EdTKTjwUs7Pf9Lkkltw71NDRtV_7iPmeDHp6byCVch8BJc9T"
ngrok.set_auth_token(NGROK_TOKEN)

# Connect to LiteLLM Proxy port 8081
# print("Starting ngrok tunnel...")
tunnel = ngrok.connect(8081)
url = tunnel.public_url

print(f"\n✅ NGROK URL: {url}")
print(f"\nUse this in your terminal:")
print(f'export ANTHROPIC_BASE_URL="{url}/v1"')
print(f'export ANTHROPIC_API_KEY="ollama"')
print(f'claude --model claude-sonnet-4-6')


✅ NGROK URL: https://ajar-guru-angelic.ngrok-free.dev

Use this in your terminal:
export ANTHROPIC_BASE_URL="https://ajar-guru-angelic.ngrok-free.dev/v1"
export ANTHROPIC_API_KEY="ollama"
claude --model claude-sonnet-4-6


### 🛠️ Fix: Ensure Model exists and LiteLLM is Running
Based on the logs, the model was missing. Let's pull it and restart the proxy.

In [ ]:
# import subprocess, time

# # 1. Ensure model is available
# print("Verifying model...")
# subprocess.run(["ollama", "pull", "qwen2.5-coder:7b"])

# # 2. Kill any old instances
# subprocess.run("pkill -f litellm", shell=True)
# time.sleep(2)

# # 3. Start LiteLLM using the config file created in the previous cell
# print("Starting LiteLLM Proxy on port 8081...")
# subprocess.Popen(
#     "litellm --config /tmp/litellm_config.yaml --port 8081 > /tmp/litellm.log 2>&1 &",
#     shell=True
# )

# # 4. Wait longer and check if the port is actually open
# time.sleep(10)
# print("Checking if port 8081 is listening...")
# !netstat -tuln | grep 8081

# print("Checking health...")
# !curl -s http://localhost:8081/health || echo 'Still not reachable'

### LiteLLM Proxy Log Analysis

Let's check the logs for the LiteLLM proxy to see if it's receiving requests and forwarding them correctly, or if there are any errors.

In [ ]:
import subprocess
# Check LiteLLM logs for errors
print("--- LiteLLM Logs ---")
!cat /tmp/litellm.log

--- LiteLLM Logs ---
INFO:     Started server process [22540]
INFO:     Waiting for application startup.

   ██╗     ██╗████████╗███████╗██╗     ██╗     ███╗   ███╗
   ██║     ██║╚══██╔══╝██╔════╝██║     ██║     ████╗ ████║
   ██║     ██║   ██║   █████╗  ██║     ██║     ██╔████╔██║
   ██║     ██║   ██║   ██╔══╝  ██║     ██║     ██║╚██╔╝██║
   ███████╗██║   ██║   ███████╗███████╗███████╗██║ ╚═╝ ██║
   ╚══════╝╚═╝   ╚═╝   ╚══════╝╚══════╝╚══════╝╚═╝     ╚═╝

INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8081 (Press CTRL+C to quit)


### Ollama Server Log Analysis

Let's check the logs for the Ollama server to identify potential issues.

In [ ]:
import subprocess

# Using tail to read the last 100 lines of the Ollama server log file
log_output = subprocess.run(['tail', '-n', '100', '/tmp/ollama_serve.log'], capture_output=True, text=True)
print(log_output.stdout)
if log_output.stderr:
    print("Error reading log file:", log_output.stderr)

time=2026-06-06T15:37:20.638Z level=INFO source=routes.go:1919 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_DEBUG:DEBUG OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://0.0.0.0:11434 OLLAMA_IGPU_ENABLE: OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/root/.ollama/models OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https://0.0.0.0 http://0.0.0.0:* h

### Verify GPU Availability

Let's check if a GPU is available in this Colab environment. If a GPU is present, we'll need to investigate why Ollama isn't utilizing it, or how to configure it to do so.

In [ ]:
import subprocess

# Check for NVIDIA GPU
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)

if "NVIDIA-SMI" in result.stdout:
    print("\n✅ GPU detected. Now we need to ensure Ollama uses it.")
    print("The Ollama logs indicate 'id=cpu', meaning it's currently using the CPU. This is the likely cause of the timeouts.")
    print("Sometimes restarting Ollama and ensuring the necessary libraries are loaded can fix this, or checking specific Ollama GPU configuration environment variables like `OLLAMA_LLM_LIBRARY`.")
else:
    print("\n❌ No NVIDIA GPU detected. Running large models on CPU will be very slow and may time out. Consider switching to a smaller model or a GPU-enabled runtime.")

FileNotFoundError: [Errno 2] No such file or directory: 'nvidia-smi'

# Test LiteLLM Proxy via Cloudflare Tunnel (No longer active as we are using ngrok)

In [ ]:
import datetime, json

def heartbeat():
    count = 0
    while True:
        count += 1
        now = datetime.datetime.now().strftime("%H:%M:%S")
        check = subprocess.run("curl -s http://localhost:11434/api/tags", shell=True, capture_output=True, text=True)
        if check.returncode != 0 or not check.stdout:
            print(f"[{now}] ⚠️ Ollama down — restarting...")
            subprocess.Popen(["/usr/local/bin/ollama", "serve"])
            time.sleep(8)
        else:
            models = json.loads(check.stdout).get('models', [])
            names = [m['name'] for m in models]
            print(f"[{now}] ✅ #{count} | Models: {names}", flush=True)
        time.sleep(5 * 60)

import threading
t = threading.Thread(target=heartbeat, daemon=True)
t.start()
print("✅ Heartbeat running in background")

In [ ]:
import datetime
print("🔁 Keep-alive active — do not stop this cell")
count = 0
while True:
    count += 1
    print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] alive #{count}", flush=True)
    time.sleep(10 * 60)

In [ ]:
# import subprocess, time, sys
# from pyngrok import ngrok

# # Kill any existing cloudflared processes
# subprocess.run("pkill cloudflared 2>/dev/null", shell=True)
# print("Killed any running cloudflared processes.")

# # Kill old ngrok tunnels
# ngrok.kill()
# time.sleep(2)

# print("Setting ngrok auth token...")
# # IMPORTANT: Replace 'YOUR_AUTHTOKEN' with your actual ngrok authtoken.
# # You can get one from your ngrok dashboard: https://dashboard.ngrok.com/get-started/your-authtoken
# # For better security, consider storing this in Colab's 'Secrets' (key icon on the left panel)
# # and loading it using `from google.colab import userdata; NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')`
# ngrok.set_auth_token("3EdTKTjwUs7Pf9Lkkltw71NDRtV_7iPmeDHp6byCVch8BJc9T")

# # Start ngrok tunnel to LiteLLM proxy on port 8081
# print("Starting ngrok tunnel to LiteLLM proxy (port 8081)...")
# tunnel = ngrok.connect(8081) # Connect to LiteLLM proxy port
# url = tunnel.public_url

# print(f"\n{'='*55}")
# print(f"✅ NGROK TUNNEL URL:\n\n    {url}\n")
# print(f"You can now use this URL in your Claude terminal:")
# print(f'  ANTHROPIC_BASE_URL="{url}/v1" \')
# print(f'  ANTHROPIC_API_KEY="ollama" \')
# print(f'  claude --model claude-sonnet-4-6')
# print(f"{'='*55}")


Lightning Notebook cell. Lets add this file in lightning ai platform with T4 GPU resource. Let make sure you add whichever port you want to use to expose on lightning AI platform. this will directly expose the Anthropic complianct API end url.

In [ ]:
import subprocess, time, os, json
import glob # For more robust path finding

print("--- Starting Combined Ollama and LiteLLM Setup ---")

# --- Ollama Installation ---
print("Fetching latest Ollama GPU release URL...")

# Fetch the latest release information from GitHub API
try:
    release_info = subprocess.run(
        "curl -s https://api.github.com/repos/ollama/ollama/releases/latest",
        shell=True, capture_output=True, text=True, check=True
    ).stdout
    latest_release = json.loads(release_info)
except (subprocess.CalledProcessError, json.JSONDecodeError) as e:
    raise RuntimeError(f"Failed to fetch latest Ollama release info: {e}")

# Try to find the standard linux-amd64 .tar.zst archive
download_url = None
ollama_archive_filename = "ollama-linux-amd64.tar.zst" # Standard AMD64 Linux build
for asset in latest_release.get('assets', []):
    asset_name = asset.get('name', '')
    asset_url = asset.get('browser_download_url', '')
    if ollama_archive_filename == asset_name:
        download_url = asset_url
        print(f"Found standard Linux AMD64 archive: {ollama_archive_filename}")
        break

if not download_url:
    raise RuntimeError(f"Could not find the standard Linux AMD64 Ollama download URL with name '{ollama_archive_filename}' in the latest release. Please check the GitHub releases manually.")

print(f"Downloading Ollama GPU version from: {download_url}")

# Download Ollama archive with retries
MAX_RETRIES = 3
for attempt in range(MAX_RETRIES):
    try:
        subprocess.run(
            f"curl -fsSL {download_url} -o /tmp/ollama.tar.zst",
            shell=True, check=True
        )
        print("Download successful!")
        break # Exit loop if download is successful
    except subprocess.CalledProcessError as e:
        print(f"Download attempt {attempt + 1}/{MAX_RETRIES} failed: {e}")
        if attempt < MAX_RETRIES - 1:
            time.sleep(5) # Wait before retrying
        else:
            raise # Re-raise the exception after all retries fail

# Install zstd using pip, as apt-get might not be available on Lightning AI
print("Attempting to install zstd via pip...")
subprocess.run("pip install python-zstandard -q", shell=True, capture_output=True)
print("zstd installation attempt via pip complete. Proceeding with extraction.")

# Define the Ollama installation root directory to a user-writable path
OLLAMA_INSTALL_ROOT = "/home/zeus/ollama_bin" # Changed to a user-writable directory
os.makedirs(OLLAMA_INSTALL_ROOT, exist_ok=True)
print(f"Created Ollama installation root directory: {OLLAMA_INSTALL_ROOT}")

# Extract Ollama to the dedicated directory
subprocess.run(f"tar --use-compress-program=unzstd -xf /tmp/ollama.tar.zst -C {OLLAMA_INSTALL_ROOT}/", shell=True, check=True)

# Attempt to find the ollama executable more robustly
OLLAMA_EXECUTABLE_PATH = None
OLLAMA_BIN_DIR = None

# Case 1: 'ollama' executable is directly in OLLAMA_INSTALL_ROOT
if os.path.exists(os.path.join(OLLAMA_INSTALL_ROOT, 'ollama')):
    OLLAMA_EXECUTABLE_PATH = os.path.join(OLLAMA_INSTALL_ROOT, 'ollama')
    OLLAMA_BIN_DIR = OLLAMA_INSTALL_ROOT
    print(f"Ollama executable found directly in root: {OLLAMA_EXECUTABLE_PATH}")
else:
    # Case 2: 'ollama' executable might be in a subdirectory (e.g., OLLAMA_INSTALL_ROOT/ollama/ollama)
    print(f"Ollama executable not found directly in {OLLAMA_INSTALL_ROOT}. Searching subdirectories...")
    ollama_exec_candidates = glob.glob(os.path.join(OLLAMA_INSTALL_ROOT, '**', 'ollama'), recursive=True)
    executable_candidates = [p for p in ollama_exec_candidates if os.path.isfile(p) and os.access(p, os.X_OK)]

    if executable_candidates:
        # Prioritize candidates directly under OLLAMA_INSTALL_ROOT or shallowest path
        executable_candidates.sort(key=lambda x: x.count(os.sep)) # Sort by path depth
        OLLAMA_EXECUTABLE_PATH = executable_candidates[0]
        OLLAMA_BIN_DIR = os.path.dirname(OLLAMA_EXECUTABLE_PATH)
        print(f"Ollama executable found at: {OLLAMA_EXECUTABLE_PATH}")
    else:
        raise RuntimeError(f"Ollama executable not found in {OLLAMA_INSTALL_ROOT} or its subdirectories after extraction.")

# Ensure we have valid paths
if not OLLAMA_EXECUTABLE_PATH or not OLLAMA_BIN_DIR:
    raise RuntimeError("Failed to determine Ollama executable path and binary directory.")

print(f"Ollama binaries directory: {OLLAMA_BIN_DIR}")

# Find the llama-server binary relative to OLLAMA_INSTALL_ROOT
# It is usually in the 'lib' directory directly under the main installation root.
LLAMA_SERVER_PATH = os.path.join(OLLAMA_INSTALL_ROOT, 'lib', 'llama-server')
if not os.path.exists(LLAMA_SERVER_PATH):
    # If not found at the most common location, broaden search within OLLAMA_INSTALL_ROOT
    print(f"llama-server not found at expected path {LLAMA_SERVER_PATH}. Searching broader within {OLLAMA_INSTALL_ROOT}...")
    llama_server_candidates = glob.glob(os.path.join(OLLAMA_INSTALL_ROOT, '**', 'llama-server'), recursive=True)
    executable_llama_server_candidates = [p for p in llama_server_candidates if os.path.isfile(p) and os.access(p, os.X_OK)]

    if executable_llama_server_candidates:
        LLAMA_SERVER_PATH = executable_llama_server_candidates[0]
        print(f"llama-server binary found at: {LLAMA_SERVER_PATH} (after broad search)")
    else:
        raise RuntimeError(f"llama-server binary not found within {OLLAMA_INSTALL_ROOT} or its subdirectories.")

subprocess.run(f"chmod +x {LLAMA_SERVER_PATH}", shell=True, check=True)
print(f"llama-server binary found and made executable at: {LLAMA_SERVER_PATH}")

# Ensure the ollama executable is executable
subprocess.run(f"chmod +x {OLLAMA_EXECUTABLE_PATH}", shell=True, check=True)

# Add the directory containing ollama and its companion binaries to PATH
os.environ["PATH"] = f"{OLLAMA_BIN_DIR}:{os.environ.get('PATH', '')}"
print(f"Added '{OLLAMA_BIN_DIR}' to PATH environment variable.")

# Verify the Ollama version using the custom executable path explicitly
v = subprocess.run([OLLAMA_EXECUTABLE_PATH, "--version"], capture_output=True, text=True, check=True)
print("✅ Ollama:", v.stdout.strip())

# Store the path to the executable and its bin directory for later use
os.environ["OLLAMA_CUSTOM_EXECUTABLE_PATH"] = OLLAMA_EXECUTABLE_PATH
os.environ["OLLAMA_CUSTOM_BIN_DIR"] = OLLAMA_BIN_DIR
os.environ["OLLAMA_LLAMA_SERVER_PATH"] = LLAMA_SERVER_PATH

# --- Ollama Server Startup ---

# Kill any existing Ollama processes to free up the port
print("Killing any existing Ollama processes...")
subprocess.run("pkill ollama 2>/dev/null", shell=True)
time.sleep(2) # Give a moment for the process to terminate

# Create a copy of the current environment variables
env_vars = os.environ.copy()

# Explicitly set the GPU-related environment variables
env_vars["OLLAMA_HOST"] = "0.0.0.0:11434"
env_vars["LD_LIBRARY_PATH"] = "/usr/local/cuda/lib64:/usr/local/nvidia/lib:/usr/local/nvidia/lib64:" + env_vars.get("LD_LIBRARY_PATH", "")

# Explicitly UNSET OLLAMA_LLM_LIBRARY if it's present, to prevent interference
if "OLLAMA_LLM_LIBRARY" in env_vars:
    print(f"Explicitly unsetting OLLAMA_LLM_LIBRARY (was: {env_vars['OLLAMA_LLM_LIBRARY']}) to prevent conflicts.")
    del env_vars["OLLAMA_LLM_LIBRARY"]

env_vars["OLLAMA_DEBUG"] = "1" # Enable debug logging for Ollama

print("Starting Ollama server with environment variables:")
for k, v in env_vars.items():
    if "OLLAMA" in k or "LD_LIBRARY_PATH" in k or "PATH" in k:
        print(f"  {k}: {v}")

# Use the executable directly, which is now correctly set in OLLAMA_CUSTOM_EXECUTABLE_PATH
ollama_exec = env_vars.get("OLLAMA_CUSTOM_EXECUTABLE_PATH")
if not ollama_exec or not os.path.exists(ollama_exec):
    raise RuntimeError(f"Ollama executable path not correctly set or found: {ollama_exec}")

ollama_bin_dir = env_vars.get("OLLAMA_CUSTOM_BIN_DIR")
if not ollama_bin_dir or not os.path.exists(ollama_bin_dir):
    raise RuntimeError(f"Ollama binary directory not correctly set or found: {ollama_bin_dir}")


print(f"Changing current working directory to {ollama_bin_dir} to start Ollama server.")
current_dir = os.getcwd()
os.chdir(ollama_bin_dir)
try:
    subprocess.Popen(
        [ollama_exec, "serve"],
        stdout=open("/tmp/ollama_serve.log", "w"),
        stderr=subprocess.STDOUT,
        env=env_vars # Pass the explicitly constructed environment variables
    )
finally:
    os.chdir(current_dir)

time.sleep(30) # Increased sleep time to allow Ollama to fully start and load libraries

# Pull only if not already cached
ollama_exec_for_commands = env_vars.get("OLLAMA_CUSTOM_EXECUTABLE_PATH")
if not ollama_exec_for_commands or not os.path.exists(ollama_exec_for_commands):
    raise RuntimeError(f"Ollama executable path for commands not correctly set or found: {ollama_exec_for_commands}")

models_command = [ollama_exec_for_commands, "list"]
print(f"Running command: {' '.join(models_command)}")
models = subprocess.run(models_command, capture_output=True, text=True, env=env_vars) # Pass env_vars here too
print(models.stdout)
if models.returncode != 0:
    print(f"Error listing models: {models.stderr}")
    raise RuntimeError("Failed to list Ollama models.")

# --- Pulling the desired model ---
TARGET_MODEL = "codellama:7b" # New target model
if TARGET_MODEL.split(':')[0] not in models.stdout:
    print(f"Pulling {TARGET_MODEL} (~4GB)... This might take a while.")
    pull_command = [ollama_exec_for_commands, "pull", TARGET_MODEL]
    subprocess.run(pull_command, check=True, env=env_vars)
else:
    print(f"✅ Model {TARGET_MODEL} already cached, skipping download")

# --- Pulling qwen2.5-coder:7b if not present (optional, can be removed if not needed) ---
# The original qwen2.5-coder:7b is still used by the alias for consistency if you still need it directly.
if "qwen2.5-coder" not in models.stdout:
    print("Pulling qwen2.5-coder:7b (~4GB) for potential alias... This might take a while.")
    pull_command = [ollama_exec_for_commands, "pull", "qwen2.5-coder:7b"]
    subprocess.run(pull_command, check=True, env=env_vars)
else:
    print("✅ Model qwen2.5-coder:7b already cached, skipping download (for alias)")


# Create an alias for the new target model for consistency if you use `ollama run` directly
# The LiteLLM config will handle mapping 'claude-sonnet-4-6' to this model.
cp_command = [ollama_exec_for_commands, "cp", TARGET_MODEL, "claude-sonnet-4-6-ollama"]
subprocess.run(cp_command, capture_output=True, check=True, env=env_vars)
print(f"✅ Ollama ready with alias 'claude-sonnet-4-6-ollama' pointing to {TARGET_MODEL}")

# Verify Ollama log for GPU usage
print("--- Checking Ollama server log for GPU status ---")
# Tail more lines to ensure we capture the final inference compute line
log_output = subprocess.run(['tail', '-n', '200', '/tmp/ollama_serve.log'], capture_output=True, text=True).stdout
print(log_output)

gpu_detected = False

# Find all 'inference compute' lines to get the latest status
inference_compute_lines = [line for line in log_output.splitlines() if 'inference compute' in line]

if inference_compute_lines:
    # The last 'inference compute' line should reflect the final decision
    last_inference_line = inference_compute_lines[-1]

    if 'library=CUDA' in last_inference_line and 'id=0' in last_inference_line and 'Tesla T4' in last_inference_line:
        gpu_detected = True

if gpu_detected:
    print("✅ Ollama is likely using the GPU (Tesla T4 detected)!")
else:
    print("❌ Ollama is using the CPU. This is expected if a GPU runtime is not available or if there are issues loading CUDA libraries.")

print("--- Ollama Setup Complete ---")

# --- LiteLLM Setup ---
print("\n--- Setting up LiteLLM Proxy ---")
# Install litellm
subprocess.run("pip install 'litellm[proxy]==1.82.4' -q", shell=True)

# Updated config to ensure the model name matches exactly what Claude terminal requests
config = f"""
model_list:
  - model_name: claude-sonnet-4-6
    litellm_params:
      model: ollama_chat/{TARGET_MODEL}
      api_base: http://localhost:11434

litellm_settings:
  drop_params: true
  set_verbose: false
"""
with open("/tmp/litellm_config.yaml", "w") as f:
    f.write(config)

# Kill existing proxy
subprocess.run("pkill litellm 2>/dev/null", shell=True)
time.sleep(2)

print("Starting LiteLLM in background on port 8081...")
subprocess.Popen(
    "litellm --config /tmp/litellm_config.yaml --port 8081 > /tmp/litellm.log 2>&1 &",
    shell=True
)

# Wait and verify health
time.sleep(5)
print("Verifying local LiteLLM health...")
health_check = subprocess.run('curl -s http://localhost:8081/health', shell=True, capture_output=True, text=True)
print(health_check.stdout.strip())
if "true" in health_check.stdout.lower():
    print("✅ LiteLLM Proxy is running and healthy at http://localhost:8081")
else:
    print("❌ LiteLLM Proxy might not be running or healthy. Check /tmp/litellm.log for details.")

print("--- Combined Ollama and LiteLLM Setup Complete ---")
print("\nYour local LiteLLM proxy is running at: `http://localhost:8081`")
print("If you were to integrate this with Lightning AI, you would typically deploy this service, and Lightning AI would provide its own public endpoint.")

## Gradio Application for Ollama (via LiteLLM Proxy)

First, let's install Gradio. Then, I'll provide the Python code for the Gradio chat application that will connect to the LiteLLM proxy, which in turn communicates with your Ollama model (`codellama:7b`).

In [ ]:
import subprocess

# Install Gradio
subprocess.run("pip install gradio -q", shell=True)
print("✅ Gradio installed.")

In [ ]:
import gradio as gr
import litellm
import os
import argparse
import time

# Ensure litellm client knows where the proxy is
# This environment variable will be set in the Docker container's startup script
litellm.api_base = os.getenv("LITELLM_API_BASE", "http://localhost:8081")
# The model name configured in LiteLLM for Ollama
MODEL_NAME = "claude-sonnet-4-6"

def chat_with_ollama(message, history):
    """Function to interact with the LiteLLM proxy."""
    messages = []
    for human, ai in history:
        messages.append({"role": "user", "content": human})
        messages.append({"role": "assistant", "content": ai})
    messages.append({"role": "user", "content": message})

    try:
        # LiteLLM client uses an OpenAI-compatible interface
        response = litellm.completion(
            model=MODEL_NAME,
            messages=messages,
            api_base=litellm.api_base,
            temperature=0.7,
            max_tokens=500
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: Could not connect to LiteLLM proxy or Ollama. Please check logs. Details: {e}"

# Parse arguments for server port and name, allowing Cloud Run to specify PORT
parser = argparse.ArgumentParser()
parser.add_argument("--server_port", type=int, default=int(os.getenv("PORT", 7860)))
parser.add_argument("--server_name", type=str, default="0.0.0.0")
args = parser.parse_args([]) # Pass empty list to args for Colab execution

# Gradio Interface
chat_interface = gr.ChatInterface(
    chat_with_ollama,
    chatbot=gr.Chatbot(height=300),
    textbox=gr.Textbox(placeholder="Ask me a question about code...", container=False, scale=7),
    title="Ollama CodeLlama Chatbot via LiteLLM",
    description="Chat with the CodeLlama model running via Ollama and LiteLLM proxy.",
    theme="soft",
    examples=[
        "Explain what a Dockerfile is.",
        "Write a Python function to reverse a string.",
        "What is the difference between a list and a tuple in Python?"
    ],
    cache_examples=True,
)

# Save the Gradio app to a file for Dockerization
app_code = '''
import gradio as gr
import litellm
import os
import argparse

# Ensure litellm client knows where the proxy is
litellm.api_base = os.getenv("LITELLM_API_BASE", "http://localhost:8081")
MODEL_NAME = "claude-sonnet-4-6"

def chat_with_ollama(message, history):
    messages = []
    for human, ai in history:
        messages.append({"role": "user", "content": human})
        messages.append({"role": "assistant", "content": ai})
    messages.append({"role": "user", "content": message})

    try:
        response = litellm.completion(
            model=MODEL_NAME,
            messages=messages,
            api_base=litellm.api_base,
            temperature=0.7,
            max_tokens=500
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {e}"

parser = argparse.ArgumentParser()
parser.add_argument("--server_port", type=int, default=int(os.getenv("PORT", 7860)))
parser.add_argument("--server_name", type=str, default="0.0.0.0")
args = parser.parse_args()

gr.ChatInterface(
    chat_with_ollama,
    chatbot=gr.Chatbot(height=300),
    textbox=gr.Textbox(placeholder="Ask me a question about code...", container=False, scale=7),
    title="Ollama CodeLlama Chatbot via LiteLLM",
    description="Chat with the CodeLlama model running via Ollama and LiteLLM proxy.",
    theme="soft",
    examples=[
        "Explain what a Dockerfile is.",
        "Write a Python function to reverse a string.",
        "What is the difference between a list and a tuple in Python?"
    ],
    cache_examples=True,
).launch(server_name=args.server_name, server_port=args.server_port, share=True)
'''
with open("chat_app.py", "w") as f:
    f.write(app_code)
print("✅ 'chat_app.py' created.")

# Start the Gradio app locally for demonstration. This will provide a temporary public URL.
print("Launching Gradio app locally...")
# Start LiteLLM proxy in the background if not already running (for local testing)
subprocess.Popen("litellm --config /tmp/litellm_config.yaml --port 8081 --host 0.0.0.0 > /tmp/litellm_app.log 2>&1 &", shell=True)
time.sleep(5) # Give LiteLLM time to start

# Set the LITELLM_API_BASE environment variable for the local Gradio app
os.environ["LITELLM_API_BASE"] = "http://localhost:8081"

# Launch Gradio in a non-blocking way for Colab, using the created file
# This will print the public URL
!python chat_app.py --server_port {args.server_port} --server_name {args.server_name}


## Deploying to Google Cloud Run with GPU

To deploy this application to Google Cloud Run with GPU support, we need to containerize it using Docker. This involves creating a `Dockerfile` that includes Ollama, LiteLLM, and your Gradio application.

### Step 1: Create a `Dockerfile` and `startup.sh`

The `Dockerfile` will build your application image, and `startup.sh` will orchestrate starting all services (Ollama, LiteLLM, Gradio) within the container. Note the use of an `nvidia/cuda` base image for GPU support.


In [ ]:
dockerfile_content = '''
# Use a CUDA-enabled base image for GPU support
FROM nvidia/cuda:12.3.2-devel-ubuntu22.04

# Set environment variables for non-interactive apt-get
ENV DEBIAN_FRONTEND=noninteractive

# Install necessary packages (curl, git, python, pip, zstd, and other build tools)
RUN apt-get update && apt-get install -y --no-install-recommends \
    curl \
    git \
    python3.10 \
    python3-pip \
    python3-venv \
    zstd \
    build-essential \
    && rm -rf /var/lib/apt/lists/*

# Set Python 3.10 as default
RUN update-alternatives --install /usr/bin/python python /usr/bin/python3.10 1
RUN update-alternatives --install /usr/bin/pip pip /usr/bin/pip3 1

# Create a working directory
WORKDIR /app

# --- Ollama Installation (adapted from previous cells) ---
# Fetch the latest Ollama GPU release URL
ARG OLLAMA_INSTALL_ROOT="/app/ollama_bin"
ENV OLLAMA_INSTALL_ROOT=${OLLAMA_INSTALL_ROOT}

RUN curl -s https://api.github.com/repos/ollama/ollama/releases/latest | \
    grep "browser_download_url" | \
    grep "ollama-linux-amd64.tar.zst" | \
    cut -d '"' -f 4 | \
    xargs -I {} curl -fsSL {} -o /tmp/ollama.tar.zst

RUN mkdir -p ${OLLAMA_INSTALL_ROOT}
RUN tar --use-compress-program=unzstd -xf /tmp/ollama.tar.zst -C ${OLLAMA_INSTALL_ROOT}/

# More robust Ollama executable path finding
ENV OLLAMA_EXECUTABLE_PATH=""
ENV OLLAMA_BIN_DIR=""

RUN if [ -f "${OLLAMA_INSTALL_ROOT}/ollama" ]; then \
        export OLLAMA_EXECUTABLE_PATH="${OLLAMA_INSTALL_ROOT}/ollama"; \
        export OLLAMA_BIN_DIR="${OLLAMA_INSTALL_ROOT}"; \
    else \
        export OLLAMA_EXECUTABLE_PATH=$(find ${OLLAMA_INSTALL_ROOT} -type f -name ollama -executable | head -n 1); \
        export OLLAMA_BIN_DIR=$(dirname "$OLLAMA_EXECUTABLE_PATH"); \
    fi; \
    if [ -z "$OLLAMA_EXECUTABLE_PATH" ]; then \
        echo "Ollama executable not found!" && exit 1; \
    fi; \
    chmod +x "$OLLAMA_EXECUTABLE_PATH"; \
    ln -s "$OLLAMA_EXECUTABLE_PATH" /usr/local/bin/ollama; \
    echo "OLLAMA_EXECUTABLE_PATH=$OLLAMA_EXECUTABLE_PATH" >> /etc/environment; \
    echo "OLLAMA_BIN_DIR=$OLLAMA_BIN_DIR" >> /etc/environment;

# Ensure llama-server is executable and linked
ENV LLAMA_SERVER_PATH=""
RUN export LLAMA_SERVER_PATH=$(find ${OLLAMA_INSTALL_ROOT} -type f -name llama-server -executable | head -n 1); \
    if [ -z "$LLAMA_SERVER_PATH" ]; then \
        echo "llama-server not found!" && exit 1; \
    fi; \
    chmod +x "$LLAMA_SERVER_PATH"; \
    echo "LLAMA_SERVER_PATH=$LLAMA_SERVER_PATH" >> /etc/environment;

# Load environment variables set above
RUN . /etc/environment

# Set OLLAMA_HOST, LD_LIBRARY_PATH, and PATH for the container runtime
ENV OLLAMA_HOST="0.0.0.0:11434"
ENV LD_LIBRARY_PATH="/usr/local/cuda/lib64:/usr/local/nvidia/lib:/usr/local/nvidia/lib64:${LD_LIBRARY_PATH}"
ENV PATH="${OLLAMA_BIN_DIR}:${PATH}"

# Install Python dependencies
COPY requirements.txt ./
RUN pip install --no-cache-dir -r requirements.txt

# Copy the Gradio application file and startup script
COPY chat_app.py ./
COPY startup.sh ./
RUN chmod +x startup.sh

# Expose the Gradio port
EXPOSE 7860

# Set the entrypoint to the startup script
ENTRYPOINT ["./startup.sh"]
'''

with open("Dockerfile", "w") as f:
    f.write(dockerfile_content)
print("✅ 'Dockerfile' created.")

requirements_content = '''
litellm[proxy]==1.82.4
gradio
httpx
"""
Note: httpx is explicitly added as litellm uses it, and sometimes it's not automatically pulled.
This ensures all dependencies for the Gradio app and LiteLLM client are present.
"""
'''
with open("requirements.txt", "w") as f:
    f.write(requirements_content)
print("✅ 'requirements.txt' created.")

startup_script_content = '''
#!/bin/bash

# Exit immediately if a command exits with a non-zero status.
set -e

echo "Starting startup.sh script..."

# The OLLAMA_HOST, LD_LIBRARY_PATH, and PATH are set in the Dockerfile already.
# Uncomment for additional debugging if needed:
# env

# Start Ollama server in background
echo "Starting Ollama server..."
ollama serve > /tmp/ollama_serve.log 2>&1 &
OLLAMA_PID=$!
echo "Ollama server started with PID: $OLLAMA_PID"

# Wait for Ollama to be ready
echo "Waiting for Ollama server to be ready..."
for i in $(seq 1 120); do # Increased wait time
    if curl -s http://localhost:11434/api/tags > /dev/null; then
        echo "Ollama server is up!"
        break
    fi
    echo "Waiting for Ollama... ($i/120)"
    sleep 1
done

if ! curl -s http://localhost:11434/api/tags > /dev/null; then
    echo "Ollama server did not start in time. Exiting."
    cat /tmp/ollama_serve.log
    exit 1
fi

# Pull the model
TARGET_MODEL="codellama:7b"
echo "Pulling model: $TARGET_MODEL"
ollama pull "$TARGET_MODEL"

# Create LiteLLM config
cat <<EOF > /tmp/litellm_config.yaml
model_list:
  - model_name: claude-sonnet-4-6
    litellm_params:
      model: ollama_chat/${TARGET_MODEL}
      api_base: http://localhost:11434

litellm_settings:
  drop_params: true
  set_verbose: false
EOF

# Start LiteLLM proxy in background
echo "Starting LiteLLM proxy..."
litellm --config /tmp/litellm_config.yaml --port 8081 --host 0.0.0.0 > /tmp/litellm.log 2>&1 &
LITELLM_PID=$!
echo "LiteLLM proxy started with PID: $LITELLM_PID"

# Wait for LiteLLM to be ready
echo "Waiting for LiteLLM proxy to be ready..."
for i in $(seq 1 60); do # Increased wait time
    if curl -s http://localhost:8081/health > /dev/null; then
        echo "LiteLLM proxy is up!"
        break
    fi
    echo "Waiting for LiteLLM... ($i/60)"
    sleep 1
done

if ! curl -s http://localhost:8081/health > /dev/null; then
    echo "LiteLLM proxy did not start in time. Exiting."
    cat /tmp/litellm.log
    exit 1
fi

# Set LITELLM_API_BASE for the Gradio app
export LITELLM_API_BASE="http://localhost:8081"

# Start Gradio app
# Google Cloud Run expects the application to listen on the port specified by the PORT environment variable.
# Gradio's default port 7860 will be used if PORT is not set by the environment (e.g., Cloud Run).
GRADIO_PORT=${PORT:-7860}
echo "Starting Gradio app on port: $GRADIO_PORT"
python chat_app.py --server_port "$GRADIO_PORT" --server_name "0.0.0.0"

# Keep services running if Gradio exits (for debugging purposes, otherwise Cloud Run will restart the container)
# wait $OLLAMA_PID $LITELLM_PID
'
with open("startup.sh", "w") as f:
    f.write(startup_script_content)
print("✅ 'startup.sh' created.")

### Step 2: Build and Deploy to Google Cloud Run (with GPU)

Now that you have your `Dockerfile`, `requirements.txt`, `chat_app.py`, and `startup.sh`, you can build your Docker image and deploy it to Google Cloud Run with GPU support.

**Prerequisites:**

1.  **Google Cloud Project:** Ensure you have a Google Cloud Project set up and billing enabled.
2.  **`gcloud` CLI:** Install and initialize the Google Cloud SDK (gcloud CLI) on your local machine.
3.  **APIs Enabled:** Enable the following APIs in your Google Cloud project:
    *   Cloud Run API
    *   Artifact Registry API (or Container Registry API if using gcr.io)
    *   Cloud Build API

**Commands to Execute (on your local machine/terminal):**

1.  **Authenticate Docker to Google Cloud:**
    ```bash
    gcloud auth configure-docker
    ```

2.  **Build your Docker image:** Replace `your-project-id` with your actual Google Cloud Project ID and `my-ollama-gradio-app` with your desired image name.
    ```bash
    PROJECT_ID="your-project-id"
    IMAGE_NAME="my-ollama-gradio-app"
    IMAGE_TAG="gcr.io/${PROJECT_ID}/${IMAGE_NAME}:latest"

    docker build -t ${IMAGE_TAG} .
    ```

3.  **Push the Docker image to Google Container Registry (GCR) or Artifact Registry:**
    ```bash
    docker push ${IMAGE_TAG}
    ```

4.  **Deploy to Google Cloud Run with GPU:**
    ```bash
    gcloud run deploy ${IMAGE_NAME} \
        --image ${IMAGE_TAG} \
        --platform managed \
        --region us-central1 \
        --allow-unauthenticated \
        --memory 16Gi \
        --cpu 4 \
        --gpu type=nvidia-tesla-t4,count=1 \
        --max-instances 1 \
        --timeout 3600s \
        --port 7860 \
        --project ${PROJECT_ID}
    ```
    *   **`--region us-central1`**: Choose a region that supports GPU on Cloud Run (e.g., `us-central1`, `us-east1`, `europe-west4`). Check [Cloud Run GPU regions](https://cloud.google.com/run/docs/configuring/gpus#locations) for the latest list.
    *   **`--gpu type=nvidia-tesla-t4,count=1`**: Specifies a Tesla T4 GPU. You might need to request a quota increase for GPUs in your project.
    *   **`--memory 16Gi`**: Ollama models can be large, ensure sufficient memory. `codellama:7b` is about 4GB, but for general operation, 16GB is a safer minimum with GPU.
    *   **`--cpu 4`**: Allocate enough CPU for Ollama and Python processes.
    *   **`--max-instances 1`**: GPU deployments are often expensive, start with 1 instance.
    *   **`--timeout 3600s`**: Increase timeout as model loading can take time.
    *   **`--port 7860`**: This is the port your Gradio app will listen on. Cloud Run will map its external port to this internal port.

After deployment, Google Cloud Run will provide you with a URL where your Gradio application is accessible. It might take a few minutes for the service to become active as the model needs to be pulled within the container on first startup.